In [ ]:
# Use a pipeline as a high-level helper
import torch
from transformers import pipeline

pipe = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

/workspaces/mlops_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use c

Bad pipe message: %s [b'0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7\r\nHost: localhost:36225\r\nUs', b'-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.']
Bad pipe message: %s [b'0.0 Safari/537.36\r\nAccept-Encoding: gzip, defla']
Bad pipe message: %s [b', br, zstd\r\nAccept-Language: it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7\r\nCache-Control: max-age=0\r\nReferer: https://shiny-', b'igma-g9pxw6rgjwwcp466.github.dev/\r\nX-Request-ID: bd29328cf51aaa00c64f18b402c0eb50\r\nX-Real-IP: 178.19']
Bad pipe message: %s [b'48.125\r\nX-Forwarded-Port: 443\r\nX-Forwarded-Sc', b'me: https\r\nX-Original-URI: /\r\nX-Scheme: https\r\nsec-fetch-site: same-site\r\nsec-fetch-mode: navigate\r\nsec', b'etch-dest: document\r\nsec-ch-ua: "Google Chro']
Bad pipe message: %s [b'";v="141", "Not?A_Brand";v="8", "Chromium";v="141"\r\nsec-ch-ua-mobile: ?0\r\nsec-ch-ua-platform: "Windows"\r\npri', b'ity: u=0, i\r\nX-Forwarded-Prot

In [2]:
test_text = "I like it so much"
pipe(test_text)

[{'label': 'positive', 'score': 0.9726189374923706}]

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhi8923shriv/sentiment-analysis-dataset")

print("Path to dataset files:", path)

100%|██████████| 54.4M/54.4M [00:02<00:00, 23.9MB/s]

Extracting files...


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/abhi8923shriv/sentiment-analysis-dataset/versions/9


In [10]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "test.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "abhi8923shriv/sentiment-analysis-dataset",
  file_path,
  pandas_kwargs={"encoding":"latin-1"}
  
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df.head()

/tmp/ipykernel_17783/1806370276.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797.0,27400.0,105.0
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative,night,31-45,Algeria,43851044.0,2381740.0,18.0
3,01082688c6,happy bday!,positive,morning,46-60,Andorra,77265.0,470.0,164.0
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive,noon,60-70,Angola,32866272.0,1246700.0,26.0


In [ ]:
# saving raw csv 
df.to_csv("../data/raw/raw_sentiment_df.csv")

In [12]:
df_processed = df[["text", "sentiment"]] 

In [13]:
df_processed

,text,sentiment
0,Last session of the day http://twitpic.com/67ezh,neutral
1,Shanghai is also really exciting (precisely -...,positive
2,"Recession hit Veronique Branquinho, she has to...",negative
3,happy bday!,positive
4,http://twitpic.com/4w75p - I like it!!,positive
...,...,...
4810,NaN,NaN
4811,NaN,NaN
4812,NaN,NaN
4813,NaN,NaN


In [14]:
df_processed.to_csv("../data/processed/df_processed.csv")

In [15]:
df_processed.shape

(4815, 2)

In [19]:
df_processed = df_processed.dropna()

In [21]:
df_processed.to_csv("../data/processed/sentiment_df_processed.csv")

In [22]:


from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset, ClassLabel, Value # Necessario per la gestione efficiente dei dati
import pandas as pd


# 1. Carica il Tokenizer
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")

# 2. Conversione DataFrame in Hugging Face Dataset (più efficiente per la tokenization)
hf_dataset = Dataset.from_pandas(df_processed)

# 3. Definisci la funzione di Tokenization e Mappatura delle Etichette
def tokenize_and_map_labels(examples):
    # Esegui la tokenization
    tokenized_inputs = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)
    
    # Esempio di mappatura: (Adatta queste etichette a quelle reali del tuo dataset!)
    label_map = {"positive": 2, "neutral": 1, "negative": 0} 
    tokenized_inputs["labels"] = [label_map[label] for label in examples['sentiment']]
    
    return tokenized_inputs

# 4. Applica la funzione all'intero dataset
tokenized_dataset = hf_dataset.map(tokenize_and_map_labels, batched=True)

print("Dataset tokenizzato e pronto per il training/retraining!")
print(tokenized_dataset)

Map: 100%|██████████| 3534/3534 [00:00<00:00, 8241.45 examples/s]

Dataset tokenizzato e pronto per il training/retraining!
Dataset({
    features: ['text', 'sentiment', '__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3534
})


In [26]:
# Importa pandas se non l'hai già fatto
import pandas as pd

# Converto le prime 5 righe in un DataFrame per l'ispezione
df_tokenized = tokenized_dataset.select(range(5)).to_pandas()

df_tokenized

,text,sentiment,__index_level_0__,input_ids,attention_mask,labels
0,Last session of the day http://twitpic.com/67ezh,neutral,0,"[0, 10285, 1852, 9, 5, 183, 1437, 2054, 640, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
1,Shanghai is also really exciting (precisely -...,positive,1,"[0, 7137, 16, 67, 269, 3571, 36, 5234, 44568, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",2
2,"Recession hit Veronique Branquinho, she has to...",negative,2,"[0, 21109, 21308, 478, 3060, 261, 5150, 2265, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
3,happy bday!,positive,3,"[0, 1372, 741, 1208, 328, 2, 1, 1, 1, 1, 1, 1,...","[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",2
4,http://twitpic.com/4w75p - I like it!!,positive,4,"[0, 2054, 640, 17137, 405, 19017, 4, 175, 73, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",2


In [27]:
# saving tokenized dataset
tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset")


Saving the dataset (1/1 shards): 100%|██████████| 3534/3534 [00:00<00:00, 256452.02 examples/s]
